# AITA dataset exploration (DATA-01)

Loads the raw Hugging Face dataset, checks its structure and shows a few normalized records.
Findings are documented in `docs/data/aita_dataset.md`.

Setup: `uv sync --group data`. Clear outputs before committing (they contain dataset text).

In [ ]:
from collections import Counter

from asshole_ai_detector.data.aita_adapter import (
    HF_REVISION,
    MAX_COMMENTS,
    iter_normalized_cases,
    load_aita_dataset,
)

raw = load_aita_dataset()  # train + test, pinned revision
print(f"revision={HF_REVISION}, rows={len(raw)}")
raw.features

## Raw data quality

In [ ]:
df = raw.to_pandas()

print("empty submission_text:", (df["submission_text"].str.strip() == "").sum())
print("missing post ID in URL:", (~df["submission_url"].str.contains("/comments/")).sum())
print("duplicate URLs:", df["submission_url"].duplicated().sum())
print("date range:", df["submission_date"].min(), "->", df["submission_date"].max())

words = df["submission_text"].str.split().str.len()
print("post words:", words.describe().round(1).to_dict())

In [ ]:
# Label distribution over all ranked comments (None = no verdict extracted).
labels = Counter()
deleted = 0
for rank in range(1, MAX_COMMENTS + 1):
    labels.update(df[f"top_comment_{rank}_classification"].fillna("None"))
    deleted += df[f"top_comment_{rank}"].isin(["[deleted]", "[removed]"]).sum()
print(labels.most_common())
print("deleted/removed comments:", deleted)

## Normalized records

Preview only: the 5-comment, 50-word and 70% rules are implemented in the preprocessing issue.

In [ ]:
cases = list(iter_normalized_cases(raw))
print(len(cases), "normalized cases")
print(cases[0].model_dump_json(indent=2)[:2000])

In [ ]:
VALID = {"NTA", "YTA", "ESH", "NAH"}
preview = Counter()
for case in cases:
    votes = Counter(c.verdict for c in case.comments if c.verdict in VALID)
    n_valid = sum(votes.values())
    if n_valid < 5:
        preview["<5 valid"] += 1
        continue
    verdict, count = votes.most_common(1)[0]  # Tie-break is ignored in this preview.
    preview[verdict if count / n_valid >= 0.70 else "DISPUTED"] += 1
preview